In [1]:

import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoTokenizer


model_name = "meta-llama/Llama-3.2-3B-Instruct"

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
from datasets import load_from_disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-amod-conversation-v2")

In [5]:
dataset["train"][0]

{'query': 'I struggle with depression as well as pretty intense mood swings throughout the month. I experience highs where I feel amazing and energetic and then lows where I lack focus, energy, and generally have a more dark outlook on my life. How can I live a more balanced life?',
 'response': "It's fun to ride the roller coaster from time to time, isn't it? :)But, it's also weary-making, and leads to drainage that no man or woman can hardly anticipate!Balance comes with proper understanding of the different bodies you possess and how they function.And, to begin, we focus upon your physical, and move right up the latter to the spiritual, and begin cleaning you out.Unbalance is an experience of blocked energies that should be naturally flowing (call them what thou mayest); when blockages are removed, what is naturally there flows, and flows beautifully...",
 'query_length': 49,
 'response_length': 92,
 'text': "\n### Input:\nPateint Query: I struggle with depression as well as pretty 

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix missing padding token issue
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
print(f"✅ Tokenizer loaded with pad_token: {tokenizer.pad_token}")

✅ Tokenizer loaded with pad_token: <|eot_id|>


In [7]:
# Load the model with full precisoin
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Full precision
).to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 57.96it/s]


In [8]:
from peft import LoraConfig, TaskType

rank_dimension = 16  # rank dimension for LoRA update matrices (smaller = more compression)
lora_alpha = 64      # lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_dropout = 0.05   # lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers (most important)
        "gate_proj", "up_proj", "down_proj",     # MLP layers
        "embed_tokens", "lm_head"                # Need for stronger adaptation
    ],
    task_type=TaskType.CAUSAL_LM,
    # bias="none"  # Add this for better performance
)

In [11]:
from trl import SFTTrainer, SFTConfig

# Configure training arguments for mental health function calling
training_args = SFTConfig(
    output_dir="/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v7",
    
    # Batch configuration - small batches with accumulation for memory efficiency
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,  # Effective batch size = 4
    
    # Learning configuration optimized for function calling
    learning_rate=1e-4,  # Conservative rate for stable training
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    weight_decay=0.1,
    
    # Training duration - more epochs for smaller specialized dataset
    num_train_epochs=10,
    
    # Evaluation and monitoring
    eval_strategy="epoch",                    # Evaluate every epoch
    save_strategy="epoch",                    # Save every epoch (changed from "no")
    save_total_limit=3,                       # Keep only 3 best checkpoints
    load_best_model_at_end=True,             # Load best model at end
    metric_for_best_model="eval_loss",       # Use validation loss as metric
    greater_is_better=False,                # Lower loss is better
    
    # Memory optimization for 3B model
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Sequence handling for mental health conversations
    # max_seq_length=2000,  # Adequate for conversation
    packing=True,  # Efficient training on variable length sequences
    dataset_text_field="text",
    
    push_to_hub=False,
)

print(f"📊 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")


📊 Effective batch size: 4


In [12]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:693: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:453: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your att

In [13]:
print("🚀 Starting training...")
trainer.train()

🚀 Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,2.148500,2.153296
2,1.876500,1.924023
3,1.551800,1.772851
4,1.368200,1.648554
5,1.093100,1.546133
6,0.839000,1.455310
7,0.676700,1.385488
8,0.486400,1.349689
9,0.448100,1.330191
10,0.413300,1.330841


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:270: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:270: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `t

TrainOutput(global_step=990, training_loss=1.1268404941366177, metrics={'train_runtime': 1327.6581, 'train_samples_per_second': 5.958, 'train_steps_per_second': 0.746, 'total_flos': 1.3695293052420096e+17, 'train_loss': 1.1268404941366177})

In [14]:
# Save the final model
print("Saving model...")
trainer.save_model()
tokenizer.save_pretrained(training_args.output_dir)

print("Training complete!")
print(f"Model saved to: {training_args.output_dir}")

Saving model...


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:270: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Training complete!
Model saved to: /home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v7
